# 外部Embedding APIでテキスト特徴量を生成

`project_name`、`project_objective`、`project_summary`をラベル付きテキストへ整形し、OpenAIまたはGeminiでベクトル化します。生成物はすべて`data/embeddings/`以下に保存します。

安全のため、このNotebookは初期状態ではdry-runしか行いません。まず費用見積りを確認し、次に5行のsmoke test、最後に全件実行の順で進めます。`science_tech_decision`はAPIへ送信しません。

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from embedding_features import (
    generate_embeddings,
    l2_normalize_embeddings,
    list_embedding_models,
    load_embeddings,
)

## 設定

OpenAIを使う場合は`OPENAI_API_KEY`、Geminiを使う場合は`GEMINI_API_KEY`を環境変数へ設定してください。キーをNotebookへ直接書かないでください。

既定はOpenAI `text-embedding-3-large`・1536次元です。Geminiへ切り替える例も下に残しています。価格は変更される可能性があるため、有料実行の直前に公式料金表を確認してください。

In [ ]:
PROVIDER = 'openai'
MODEL = 'text-embedding-3-large'
EMBEDDING_DIM = 1536  # Noneならモデル既定次元
PRICE_PER_MILLION_TOKENS = 0.13

# Geminiを使う場合:
# PROVIDER = 'gemini'
# MODEL = 'gemini-embedding-2'
# EMBEDDING_DIM = 1536
# PRICE_PER_MILLION_TOKENS = 0.20

TEXT_COLS = ['project_name', 'project_objective', 'project_summary']
TARGET_COL = 'science_tech_decision'
OUTPUT_ROOT = PROJECT_ROOT / 'data' / 'embeddings'
BATCH_SIZE = 32
MAX_BUDGET_USD = 20.0
MAX_INPUT_TOKENS = 8000

RUN_SMOKE_API = False  # 5行だけ実APIを呼ぶときTrue
RUN_FULL_API = False   # 全件を実APIで生成するときTrue

In [ ]:
train = pd.read_csv(PROJECT_ROOT / 'input' / 'train.csv')
test = pd.read_csv(PROJECT_ROOT / 'input' / 'test.csv')

display(train[TEXT_COLS].head())
print('train:', train.shape, 'test:', test.shape)

## 1. dry-run（API呼び出しなし）

行数、空テキスト、truncate候補、推定token数、推定費用を確認します。ここではAPI clientも作成されません。

In [ ]:
common_kwargs = dict(
    provider=PROVIDER,
    model=MODEL,
    output_root=OUTPUT_ROOT,
    text_cols=TEXT_COLS,
    target_col=TARGET_COL,
    embedding_dim=EMBEDDING_DIM,
    batch_size=BATCH_SIZE,
    max_input_tokens=MAX_INPUT_TOKENS,
    price_per_million_tokens=PRICE_PER_MILLION_TOKENS,
    max_budget_usd=MAX_BUDGET_USD,
)

train_dry_run = generate_embeddings(train, split='train', dry_run=True, **common_kwargs)
test_dry_run = generate_embeddings(test, split='test', dry_run=True, **common_kwargs)
dry_run_reports = pd.DataFrame([train_dry_run['report'], test_dry_run['report']])
display(dry_run_reports)
print('train + test rows:', int(dry_run_reports['total_rows'].sum()))
print('train + test estimated tokens:', int(dry_run_reports['total_estimated_tokens'].sum()))
TOTAL_ESTIMATED_COST_USD = float(dry_run_reports['estimated_cost_usd'].sum())
print('train + test estimated cost (USD):', TOTAL_ESTIMATED_COST_USD)
print('within total budget:', TOTAL_ESTIMATED_COST_USD <= MAX_BUDGET_USD)

利用できるembeddingモデル名をアカウント側で確認したい場合だけ、次のコメントを外します。この確認にもAPIキーとネットワーク接続が必要です。指定モデルが利用不能でも自動fallbackはしません。

In [ ]:
# list_embedding_models(PROVIDER)

## 2. 5行のsmoke test

dry-runと料金を確認した後、`RUN_SMOKE_API=True`にして実行します。保存先は本番と分離した`data/embeddings_smoke/`です。

In [ ]:
smoke_result = None
if RUN_SMOKE_API:
    smoke_kwargs = {**common_kwargs, 'output_root': PROJECT_ROOT / 'data' / 'embeddings_smoke'}
    smoke_result = generate_embeddings(
        train.head(5),
        split='train_smoke',
        dry_run=False,
        **smoke_kwargs,
    )
    assert smoke_result['embeddings'].shape[0] == min(5, len(train))
    assert np.isfinite(smoke_result['embeddings']).all()
    display(smoke_result['metadata'])
else:
    print('Smoke API call is disabled. Set RUN_SMOKE_API=True after checking the dry-run.')

## 3. train/test全件生成

`RUN_FULL_API=True`の場合だけ有料APIを呼びます。batchごとにshardと進捗を保存するため、停止後に同じ設定で再実行すると完了済みbatchを検証して再利用します。完成済みcacheがあればAPIキーなしでも読み込めます。

In [ ]:
train_result = None
test_result = None
if RUN_FULL_API:
    if TOTAL_ESTIMATED_COST_USD > MAX_BUDGET_USD:
        raise RuntimeError('Estimated train + test cost exceeds configured budget.')
    train_result = generate_embeddings(train, split='train', dry_run=False, **common_kwargs)
    test_result = generate_embeddings(test, split='test', dry_run=False, **common_kwargs)
    print('train:', train_result['embeddings'].shape, train_result['cache_dir'])
    print('test :', test_result['embeddings'].shape, test_result['cache_dir'])
else:
    print('Full API call is disabled. Set RUN_FULL_API=True only after the smoke test succeeds.')

## 読み込み・整合性確認・L2正規化

`generate_embeddings`の戻り値には保存先が含まれます。`load_embeddings`へ元DataFrameを渡すと、行数だけでなく`project_id`と元indexの順序も検証します。保存するのはraw embeddingで、正規化が必要なモデルだけ別途コピーを作ります。

In [ ]:
if train_result is not None:
    train_embeddings, train_metadata = load_embeddings(
        train_result['cache_dir'],
        split='train',
        expected_df=train,
    )
    train_embeddings_l2 = l2_normalize_embeddings(train_embeddings)
    assert train_embeddings.shape[0] == len(train_metadata) == len(train)
    assert np.isfinite(train_embeddings_l2).all()
    print(train_embeddings.shape, train_embeddings.dtype)